# The second model: how the players have been playing

Notebook 04's agent model scored 51.4% against a 55.5% baseline. Notebook 05 showed player
form is worth building properly. This is the real version: the feature lives in
[`src/form.py`](../src/form.py), older maps can count for less, and the cut-off is proved by
[`test_form.py`](../test_form.py).

## The one thing that could ruin this

Player form is built from statistics recorded *during* maps. They're only legitimate as a
record of **earlier** matches. Off by one match and the result leaks into its own prediction:
the score looks wonderful and means nothing. That's how the public project in README's
"Similar projects" reaches 93%.

`test_form.py` checks four things, and all four pass:

1. **By hand:** one real map's averages worked out manually match the code.
2. **No reaching backwards:** features rebuilt from a cut-short season don't change.
3. **Nothing known at the start:** the season's first match comes back blank.
4. **Fading works:** with a half-life set, recent maps count for more.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from dataset import build_map_table, season_folds, split_season
from features import build_features, feature_columns
from form import player_form

maps = build_map_table()
learn, test = split_season(maps)

# The column list is worked out once from the WHOLE season and used everywhere.
# Without this the two halves come back different shapes -- the map pool rotated
# mid-season -- and a model fitted on one could not score the other.
COLUMNS = feature_columns(maps)
X_agents, y = build_features(learn, COLUMNS)
agent_columns = [c for c in X_agents.columns if c.startswith("agent_")]

print(f"learning half: {len(learn):,} maps, described by {len(COLUMNS)} columns")
print(f"test half:     {len(test):,} maps  (not touched in this notebook)")

learning half: 756 maps, described by 39 columns
test half:     516 maps  (not touched in this notebook)


## 1. How to measure success

**Accuracy is the headline**, which is defensible here: it misleads when one outcome dominates,
and ours split 51/49. It's also what the baselines use.

**Log loss is reported alongside**, because accuracy throws away the probability: "65%" and
"51%" score the same when both are right. Log loss rewards confident-and-right and punishes
confident-and-wrong. Lower is better.

| Log loss | Meaning |
|---|---|
| **0.693** | saying 50/50 to every map: knowing nothing |
| below 0.693 | the probabilities carry real information |
| above 0.693 | worse than knowing nothing |

## 2. How fast should old performances fade?

`half_life` is counted in maps: at 10, a performance ten maps ago counts half as much as the
latest one. `None` means no fading.

In [2]:
def fold_scores(X, columns):
    """Accuracy and log loss across the five folds, using only these columns."""
    accuracy, losses = [], []
    for learn_rows, check_rows in season_folds(learn):
        x_train, x_check = X.loc[learn_rows, columns], X.loc[check_rows, columns]

        # scaled using the training part only -- using the whole season's
        # averages would leak information backwards
        middle, spread = x_train.mean(), x_train.std().replace(0, 1)
        x_train, x_check = (x_train - middle) / spread, (x_check - middle) / spread

        model = LogisticRegression(max_iter=5000).fit(x_train, y.loc[learn_rows])
        accuracy.append(model.score(x_check, y.loc[check_rows]))
        losses.append(log_loss(y.loc[check_rows], model.predict_proba(x_check)[:, 1]))
    return np.array(accuracy), np.array(losses)


baseline_accuracy = []
for _, check_rows in season_folds(learn):
    picked_by_a = learn.loc[check_rows, "map_picked_by"].map({"team_a": 1, "team_b": 0})
    baseline_accuracy.append((picked_by_a.fillna(1) == y.loc[check_rows]).mean())
baseline_accuracy = np.array(baseline_accuracy)

print(f"baseline: {baseline_accuracy.mean() * 100:.1f}%\n")

forms = {}
print(f"{'fading':16} {'accuracy':>9} {'worst':>7} {'best':>6} {'log loss':>10}   beats baseline")
for half_life in [None, 40, 20, 10, 7, 5, 3]:
    forms[half_life] = player_form(learn, half_life=half_life).fillna(0.0)
    X = pd.concat([X_agents, forms[half_life]], axis=1)
    accuracy, losses = fold_scores(X, ["rating_diff", "map_picked_by_a"])
    label = "none" if half_life is None else f"half-life {half_life}"
    print(f"{label:16} {accuracy.mean() * 100:8.1f}% {accuracy.min() * 100:6.1f}% "
          f"{accuracy.max() * 100:5.1f}% {losses.mean():10.3f}   "
          f"{int((accuracy > baseline_accuracy).sum())}/5")

baseline: 55.5%

fading            accuracy   worst   best   log loss   beats baseline


none                 57.8%   56.0%  59.6%      0.680   4/5


half-life 40         57.8%   56.0%  60.3%      0.680   3/5


half-life 20         57.9%   55.2%  60.3%      0.679   4/5


half-life 10         58.5%   55.2%  61.8%      0.679   4/5


half-life 7          59.0%   55.2%  64.1%      0.678   4/5


half-life 5          59.7%   56.5%  64.1%      0.678   4/5


half-life 3          59.6%   58.1%  61.8%      0.680   4/5


**Recent form counts for more.** Accuracy climbs steadily from 57.8% with no fading to about
59.7% at three to five maps. The trend across seven settings matters more than any one row,
and anything from three to ten gives 58.5–59.7%, so the choice isn't a knife edge.
**A half-life of 5 is used from here on.**

Seven settings were tried and one kept, which flatters a little. The defence is that the
whole range works, and the test half gets scored once, at the end.

## 3. Which statistics actually help?

In [3]:
HALF_LIFE = 5
form = forms[HALF_LIFE]
X_all = pd.concat([X_agents, form], axis=1)

form_columns = [c for c in form.columns if c.endswith("_diff")]

candidates = {
    "picker only (notebook 04's best)": ["map_picked_by_a"],
    "agents + map + picker (notebook 04)": COLUMNS,
    "rating only": ["rating_diff"],
    "combat score only": ["acs_diff"],
    "rating + picker": ["rating_diff", "map_picked_by_a"],
    "combat score + picker": ["acs_diff", "map_picked_by_a"],
    "rating + combat score + picker": ["rating_diff", "acs_diff", "map_picked_by_a"],
    "all four stats + picker": form_columns + ["map_picked_by_a"],
    "rating + picker + agents": ["rating_diff", "map_picked_by_a"] + agent_columns,
}

print(f"{'':36} {'accuracy':>9} {'worst':>7} {'best':>6} {'log loss':>10}   beats baseline")
for name, columns in candidates.items():
    accuracy, losses = fold_scores(X_all, columns)
    print(f"{name:36} {accuracy.mean() * 100:8.1f}% {accuracy.min() * 100:6.1f}% "
          f"{accuracy.max() * 100:5.1f}% {losses.mean():10.3f}   "
          f"{int((accuracy > baseline_accuracy).sum())}/5")

                                      accuracy   worst   best   log loss   beats baseline
picker only (notebook 04's best)         54.4%   50.0%  58.9%      0.691   0/5
agents + map + picker (notebook 04)      51.1%   47.3%  54.3%      0.760   2/5


rating only                              58.1%   54.4%  63.1%      0.681   2/5
combat score only                        55.2%   51.1%  61.5%      0.681   2/5
rating + picker                          59.7%   56.5%  64.1%      0.678   4/5
combat score + picker                    59.6%   52.5%  67.9%      0.678   5/5


rating + combat score + picker           59.7%   51.6%  64.9%      0.678   5/5
all four stats + picker                  51.2%   37.9%  56.9%      0.723   2/5
rating + picker + agents                 54.2%   50.7%  59.5%      0.733   2/5


Notebook 04 reported 51.4% for the agent model; here it's **51.1%**. Same model, but here the
columns are put on a common scale first, which nudges a regularised model slightly. Both are
correct.

- **Less is more.** Rating, combat score, damage and KAST all measure how well a player has
  been playing, so piling them in adds noise. With 756 maps, every extra column costs.
- **The agent columns still hurt**, by several points. That's the fourth separate measurement
  saying so.

## 4. Stronger or weaker regularisation

Small `C` squeezes the coefficients hard; large `C` lets them run free.

In [4]:
BEST = ["rating_diff", "map_picked_by_a"]

print(f"{'C':>8} {'accuracy':>9} {'worst':>7} {'best':>6} {'log loss':>10}")
for C in [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 100.0]:
    accuracy, losses = [], []
    for learn_rows, check_rows in season_folds(learn):
        x_train, x_check = X_all.loc[learn_rows, BEST], X_all.loc[check_rows, BEST]
        middle, spread = x_train.mean(), x_train.std().replace(0, 1)
        x_train, x_check = (x_train - middle) / spread, (x_check - middle) / spread
        model = LogisticRegression(C=C, max_iter=5000).fit(x_train, y.loc[learn_rows])
        accuracy.append(model.score(x_check, y.loc[check_rows]))
        losses.append(log_loss(y.loc[check_rows], model.predict_proba(x_check)[:, 1]))
    accuracy = np.array(accuracy)
    print(f"{C:8} {np.mean(accuracy) * 100:8.1f}% {accuracy.min() * 100:6.1f}% "
          f"{accuracy.max() * 100:5.1f}% {np.mean(losses):10.3f}")

       C  accuracy   worst   best   log loss
    0.01     59.1%   52.2%  63.4%      0.682


     0.1     60.0%   56.5%  64.1%      0.678


     0.3     59.5%   56.5%  64.1%      0.678
     1.0     59.7%   56.5%  64.1%      0.678


     3.0     59.9%   56.5%  64.1%      0.678


    10.0     59.9%   56.5%  64.1%      0.678
   100.0     59.9%   56.5%  64.1%      0.678


## 5. The two models side by side

In [5]:
comparison = []
for name, columns in [
    ("without player stats (notebook 04)", COLUMNS),
    ("with player form", BEST),
]:
    accuracy, losses = fold_scores(X_all, columns)
    comparison.append({
        "model": name,
        "accuracy": f"{accuracy.mean() * 100:.1f}%",
        "worst fold": f"{accuracy.min() * 100:.1f}%",
        "best fold": f"{accuracy.max() * 100:.1f}%",
        "log loss": round(losses.mean(), 3),
        "beats baseline": f"{int((accuracy > baseline_accuracy).sum())}/5",
    })

comparison.insert(0, {
    "model": "baseline: who picked the map",
    "accuracy": f"{baseline_accuracy.mean() * 100:.1f}%",
    "worst fold": f"{baseline_accuracy.min() * 100:.1f}%",
    "best fold": f"{baseline_accuracy.max() * 100:.1f}%",
    "log loss": "-",
    "beats baseline": "-",
})

print(pd.DataFrame(comparison).to_string(index=False))
print("\nlog loss reference: saying 50/50 every time scores 0.693")

                             model accuracy worst fold best fold log loss beats baseline
      baseline: who picked the map    55.5%      50.0%     58.9%        -              -
without player stats (notebook 04)    51.1%      47.3%     54.3%     0.76            2/5
                  with player form    59.7%      56.5%     64.1%    0.678            4/5

log loss reference: saying 50/50 every time scores 0.693


## What this notebook found

| | Accuracy | Log loss |
|---|---|---|
| Baseline: who picked the map | 55.5% | — |
| Without player stats (notebook 04) | 51.1% | 0.760 |
| **With player form** | **59.7%** | **0.678** |

1. **Player form predicts the winner; agent picks don't.** With form, it beats the baseline
   on four folds of five.
2. **Recent form matters more than old**, and any half-life from three to ten maps works.
3. **Fewer columns win:** two beat four, and four beat thirty-nine.
4. **The probabilities are only slightly informative** (0.678 against 0.693). It picks the
   winner more often than chance but isn't confident, which is honest for close matches.
5. **The agent model's log loss, 0.760, is worse than knowing nothing.** Accuracy called it
   mediocre; log loss shows it's misleading, because it commits to answers with no grounds.

**Still to do:** confirm on the test half, once. Rating is missing for much of China early in
the season (fixed in notebook 09). Roster changes and opponent strength are invisible.